# π0 LoRA 학습 파라미터 설정 가이드

> **목적:** `config/training/pi0_piper_lora.yaml`에 있는 값을 보고, 무엇은 고정해야 하고 무엇은 조심해서 조절할 수 있으며 무엇은 별도 production profile로 구현해야 하는지 이해한다.

이 노트북은 모델 학습 노트북이 아니다. GPU를 초기화하거나 π0 weight를 읽거나 checkpoint를 저장하지 않는다. 첫 코드 셀은 production의 `piper_vla.training.settings.load_pi0_settings()`를 직접 호출해 YAML을 검증하고, 나머지 코드는 값을 화면에 보여주거나 명령 문자열을 만들기만 한다.

> **실행 전:** 이 노트북은 새 kernel에서 연다. 첫 코드 셀은 `jax`와 `openpi`가 이미 import돼 있으면 중단한다. 따라서 production loader가 GPU framework 없이 동작한다는 사실을 분명하게 확인할 수 있다.

> **학습 중 주의:** 다른 terminal에서 학습이 실행 중일 때는 YAML을 수정하지 않는다. Markdown은 읽을 수 있지만 실제 학습 설정 변경과 새 학습 명령은 현재 process가 끝난 뒤 수행한다.

## 먼저 기억할 세 분류

| 분류 | 뜻 | 기존 run에서 변경 | 대표 예 |
|---|---|---|---|
| **고정 계약** | dataset·checkpoint·추론이 같은 뜻과 shape를 공유하기 위한 규칙 | 금지 | action 7D, horizon 50, delta mask, 모델 variant |
| **구조 비변경 튜닝** | model parameter tree는 같지만 성능·메모리·재현성에는 영향을 줄 수 있는 실행값 | 항목별 판단 | 목표 step, batch, worker, log/save 간격 |
| **구조 실험** | parameter tree, optimizer state 또는 데이터 의미가 바뀌는 변경 | 금지, 별도 profile+새 run 필요 | vision 동결, LoRA rank, action horizon 변경 |

`안전 튜닝`은 어떤 숫자를 넣어도 안전하다는 뜻이 아니다. Loader의 범위 검사를 통과해도 큰 batch는 OOM을 만들 수 있고 너무 큰 worker 수는 host RAM을 고갈시킬 수 있다.

표에 붙는 근거 표시는 다음과 같다.

- `[코드]`: 현재 production 또는 pinned OpenPI에 실제로 고정된 값
- `[측정]`: 이 PC에서 step 1000까지 직접 관측한 값
- `[실험 후보]`: 아직 결과가 보장되지 않아 별도 구현·run으로 비교해야 하는 값

## 두 실행 파일의 역할

- `scripts/training/train_from_config.py`: 사람이 사용하는 YAML 기반 wrapper다. 이 wrapper를 사용할 때 `training`과 `runtime` 값의 source는 YAML이다.
- `scripts/training/train.py`: 검증된 내부 trainer이자 기존 명령 호환 CLI다. Wrapper가 YAML을 검사한 뒤 이 파일의 CLI 인자로 변환해 전달한다.

## 이 노트북의 실행 규칙

1. 새 kernel에서 위에서 아래로 실행한다.
2. 코드 셀은 설정을 읽고 문자열을 만드는 일만 한다.
3. `jax`, `openpi`, `torch`, model loader, checkpoint manager는 import하지 않는다.
4. 파일을 저장하거나 directory를 만들지 않는다.
5. 실제 학습 변경은 새 run인지 resume인지 먼저 결정한 뒤 terminal 명령으로 수행한다.


In [ ]:
from pathlib import Path
import hashlib
import sys

import yaml


# 이 프로젝트의 절대 workspace 경로다.
WORKSPACE_ROOT = Path("/home/pc/vla_ws").resolve()

# production YAML loader가 들어 있는 Python source 경로다.
WORKSPACE_SOURCE_ROOT = WORKSPACE_ROOT / "src"

# sys.path에서 비교할 workspace source 문자열이다.
WORKSPACE_SOURCE_TEXT = str(WORKSPACE_SOURCE_ROOT)

if WORKSPACE_SOURCE_TEXT not in sys.path:
    sys.path.insert(0, WORKSPACE_SOURCE_TEXT)

# production loader를 부르기 전에 없어야 하는 GPU framework module prefix다.
FORBIDDEN_MODULE_PREFIXES = ("jax", "openpi")

# 이 kernel에 이미 올라와 있는 금지 module 이름이다.
LOADED_ACCELERATOR_MODULES_BEFORE = tuple(
    sorted(
        module_name
        for module_name in sys.modules
        if any(
            module_name == prefix or module_name.startswith(f"{prefix}.")
            for prefix in FORBIDDEN_MODULE_PREFIXES
        )
    )
)

if LOADED_ACCELERATOR_MODULES_BEFORE:
    raise RuntimeError(
        "새 kernel에서 실행해야 합니다. 이미 GPU framework가 import됐습니다: "
        f"{LOADED_ACCELERATOR_MODULES_BEFORE[:10]}"
    )

# JAX/OpenPI를 import하지 않는 production strict YAML loader다.
from piper_vla.training.settings import load_pi0_settings


# production π0 학습 설정 파일의 절대 경로다.
CONFIG_PATH = WORKSPACE_ROOT / "config" / "pi0_piper_lora.yaml"

# Production과 같은 schema·locked mirror 검사를 통과한 resolved 설정이다.
PRODUCTION_SETTINGS = load_pi0_settings(CONFIG_PATH, WORKSPACE_ROOT)

# 사람이 확인할 설정 원문 byte다. 읽기만 하며 수정하지 않는다.
CONFIG_BYTES = CONFIG_PATH.read_bytes()

# Resume 기록에 남길 수 있는 현재 YAML SHA-256이다.
CONFIG_SHA256 = hashlib.sha256(CONFIG_BYTES).hexdigest()

# 다음 셀에서 최소한의 raw key/value를 펼쳐 보여주기 위한 mapping이다.
CONFIG_DATA = yaml.safe_load(CONFIG_BYTES.decode("utf-8"))

if not isinstance(CONFIG_DATA, dict):
    raise TypeError("YAML 최상위 값은 mapping이어야 합니다.")

# Production loader 호출 뒤 새로 import된 금지 module 이름이다.
LOADED_ACCELERATOR_MODULES_AFTER = tuple(
    sorted(
        module_name
        for module_name in sys.modules
        if any(
            module_name == prefix or module_name.startswith(f"{prefix}.")
            for prefix in FORBIDDEN_MODULE_PREFIXES
        )
    )
)

if LOADED_ACCELERATOR_MODULES_AFTER:
    raise AssertionError(
        "Production YAML loader가 GPU framework를 import했습니다: "
        f"{LOADED_ACCELERATOR_MODULES_AFTER[:10]}"
    )

print("Config path        :", PRODUCTION_SETTINGS.config_path)
print("Config SHA-256     :", CONFIG_SHA256)
print("Dataset            :", PRODUCTION_SETTINGS.paths.dataset_root)
print("Asset id           :", PRODUCTION_SETTINGS.dataset.asset_id)
print("Target step        :", PRODUCTION_SETTINGS.training.num_train_steps)
print(
    "Batch / workers    :",
    PRODUCTION_SETTINGS.training.batch_size,
    "/",
    PRODUCTION_SETTINGS.training.num_workers,
)
print(
    "Log / save / keep  :",
    PRODUCTION_SETTINGS.training.log_interval,
    "/",
    PRODUCTION_SETTINGS.training.save_interval,
    "/",
    PRODUCTION_SETTINGS.training.keep_period,
)
print("JAX memory fraction:", PRODUCTION_SETTINGS.runtime.jax_memory_fraction)
print("Raw top-level keys :", tuple(CONFIG_DATA))
print("JAX/OpenPI import  : NOT USED")
print("Weight/checkpoint  : NOT USED")
print("File write         : NOT USED")
print("PASS: production strict YAML loader")
print("PASS: locked YAML mirrors match expected baseline")
print("NOTE: actual OpenPI trainer/model validation was not run in this cell")


## 1. Production loader로 YAML 읽기

이 노트북은 별도의 간이 validator를 만들지 않는다. 첫 코드 셀에서 실제 wrapper와 같은 `piper_vla.training.settings.load_pi0_settings()`를 호출한다. 따라서 노트북에서 통과한 schema와 production이 해석하는 schema가 달라지는 문제를 피한다.

Production loader가 확인하는 항목:

| 검사 | 실제 guardrail | 실패 예 |
|---|---|---|
| Schema | top-level과 각 section key가 정확히 일치 | 누락 key, 알 수 없는 key |
| 타입·최솟값 | bool을 정수로 받지 않고 양수·0 이상 조건 검사 | `batch_size: true`, step 0 |
| 경로 | 모든 경로가 workspace 안에 있어야 함 | `../../outside` |
| 식별자 | asset ID가 안전한 단일 이름이어야 함 | slash가 포함된 ID |
| Locked mirror | optimizer, LR, model contract가 baseline과 exact match | horizon 40, peak LR 변경 |
| Runtime | `jax_memory_fraction`이 유한하고 `0 < value <= 1` | 0, 1.1, NaN |

### `Locked contracts: PASS`의 정확한 뜻

이 문구는 YAML의 `optimizer`, `lr_schedule`, `model_contract`가 `pi0_settings.py`에 기록된 baseline mirror와 정확히 같다는 뜻이다. 이 단계에서는 JAX/OpenPI를 import하지 않으므로 실제 model parameter나 pinned OpenPI TrainConfig를 만든 것은 아니다.

실제 학습으로 위임되면 `pi0_training.build_pi0_train_config()`와 `validate_pi0_train_config()`가 pinned OpenPI profile, π0/π0.5 구분, model variant, action shape, freeze filter, EMA, path를 다시 별도로 검사한다. 즉 검증은 다음 두 층이다.

```text
PyYAML production loader: schema + 값 범위 + locked YAML mirror
            ↓
OpenPI trainer validator: 실제 pinned TrainConfig + model/data/checkpoint 계약
```

첫 셀의 `yaml.safe_load()`는 다음 표를 보여주기 위한 최소 raw display에만 쓴다. 합격 여부를 결정하는 코드는 `load_pi0_settings()`다.


In [ ]:
def flatten_config(value, prefix=""):
    """중첩된 YAML 값을 점으로 연결한 key와 값의 목록으로 펼친다."""

    flattened = []
    if isinstance(value, dict):
        if not value:
            flattened.append((prefix, {}))
        for key, child_value in value.items():
            child_prefix = f"{prefix}.{key}" if prefix else str(key)
            flattened.extend(flatten_config(child_value, child_prefix))
    elif isinstance(value, list):
        if not value:
            flattened.append((prefix, []))
        for index, child_value in enumerate(value):
            child_prefix = f"{prefix}[{index}]"
            flattened.extend(flatten_config(child_value, child_prefix))
    else:
        flattened.append((prefix, value))
    return flattened


# 화면에 표시할 모든 YAML leaf 값이다.
FLAT_CONFIG_VALUES = tuple(flatten_config(CONFIG_DATA))

for config_key, config_value in FLAT_CONFIG_VALUES:
    print(f"{config_key:<48} = {config_value!r}")

print()
print("Leaf values     :", len(FLAT_CONFIG_VALUES))
print("PASS: current YAML values displayed")


## 2. 고정 계약 — 기존 `r001`에서 바꾸지 않는 값

고정 계약은 단순 취향값이 아니다. dataset adapter가 만든 숫자의 뜻, π0 parameter shape, checkpoint restore, 추론 서버의 역변환이 모두 같은 규칙을 사용하게 만드는 약속이다.

| 항목 | 현재값 `[코드]` | 바꾸면 생기는 일 | 예상 증상 | VRAM·속도 |
|---|---|---|---|---|
| 모델 | π0, `pi05=False`, `pi0_base` | 다른 parameter tree가 됨 | weight load 또는 validator 실패 | 다른 모델 비용은 현재 근거로 예측 불가 |
| 언어·영상 expert | `gemma_2b_lora` | checkpoint와 freeze filter가 달라짐 | restore shape 오류 | variant에 따라 달라짐 |
| action expert | `gemma_300m_lora` | action model parameter가 달라짐 | restore shape 오류 | variant에 따라 달라짐 |
| Piper action 의미 | 7D, joint 1~6 delta, gripper absolute | 같은 숫자가 다른 물리 명령이 됨 | loss는 정상인데 로봇이 잘못 움직일 수 있음 | 거의 영향 없음 |
| action horizon | 50 step, 20Hz에서 2.5초 | target·JIT·model shape가 바뀜 | adapter 검증 또는 restore 실패 | 늘리면 action compute·activation 증가 `[추정]` |
| model action/state dim | 32D, 실제 앞 7D 사용 | π0 고정 shape가 바뀜 | model validator 실패 | 늘리면 compute 증가 `[추정]` |
| 이미지 | active 2개, 224×224 | vision 입력 계약 변경 | JIT 재compile, rollout 실패 가능 | 해상도가 커지면 vision 비용 크게 증가 `[추정]` |
| prompt | 최대 48 token | tokenizer/model 입력 길이 변경 | 잘림 또는 shape 오류 | 길면 attention 비용 증가 `[추정]` |
| normalization | 7D mean/std, dataset별 asset | 학습과 추론 숫자 단위가 달라짐 | resume asset 불일치 또는 위험한 action | VRAM 영향 없음 |
| freeze·precision | base LLM 동결 bf16, LoRA·vision·projection 학습 fp32 | optimizer state 구조 변경 | resume 실패 또는 OOM | EMA를 켜면 약 7GiB logical 추가 |
| EMA / FSDP | `None` / 1 device | model 사본 또는 device topology 변경 | OOM·restore·multi-device 오류 | EMA는 큰 증가, FSDP는 별도 다중 GPU 설계 필요 |

### `r001` resume에서 금지하는 변경

- 모델 variant, LoRA rank, freeze filter
- action horizon·dimension·delta/gripper 규칙
- 카메라 mapping·활성 mask·이미지 규격
- dataset 의미 또는 normalization asset
- EMA, precision, FSDP topology

이 값은 현재 YAML에서 locked mirror이므로 파일만 고치면 production loader가 즉시 거부한다. **새 run 이름만 만드는 것으로도 부족하다.** 별도 production profile, model/data transform과 validator를 구현한 뒤 새 config·새 run을 만들고, 필요한 경우 새 asset ID와 normalization stats, abstract parameter audit, 첫 step, checkpoint restore 검증을 다시 수행한다.


## 3. 구조 비변경 튜닝 — Loader 통과와 학습 안전은 다르다

`training`과 `runtime` 값은 model parameter tree를 바꾸지 않는다. 그래서 구조 변경보다 조절하기 쉽지만, **모든 숫자가 안전하다는 뜻은 아니다.** Strict loader는 타입·최솟값·workspace 경계 같은 명백한 오류를 막을 뿐, 현재 GPU에서 OOM이 나는 batch나 너무 많은 worker까지 예측하지 않는다.

| 파라미터 | 현재값 | Loader hard guardrail | 늘리면 | 이상 증상·비용 | 현재 `r001` 원칙 |
|---|---:|---|---|---|---|
| `num_train_steps` | 30,000 | 정수, 1 이상 | 더 많은 sample, 총시간 증가 | training loss만으로 과적합 판단 불가 | `--target-step`만 사용 |
| `batch_size` | 1 | 정수, 1 이상 | activation·sample/step 증가 | OOM, 새 JIT, 비교 sample 수 변화 | 변경 후 resume 금지 |
| `num_workers` | 0 | 정수, 0 이상 | decode 병렬화, CPU/RAM 증가 | hang, shared-memory 오류, data time 악화 | YAML 그대로 유지 |
| `log_interval` | 10 | 정수, 1 이상 | sync 감소, 문제 발견 지연 | 너무 크면 NaN 확인이 늦음 | YAML 그대로 유지 |
| `save_interval` | 1,000 | 정수, 1 이상 | 저장 pause 감소, 복구 간격 증가 | 너무 작으면 I/O·host RAM spike | YAML 그대로 유지 |
| `keep_period` | 5,000 | 정수, 1 이상 | 장기 보존 checkpoint 수 감소 | 너무 작으면 디스크 증가 | YAML 그대로 유지 |
| `seed` | 42 | 정수, 0 이상 | 숫자 크기 자체 의미 없음 | shuffle/noise와 결과가 달라짐 | 변경 후 resume 금지 |
| `jax_memory_fraction` | 0.80 | 유한수, `0 < value <= 1` | JAX pool 상한 증가 | 너무 낮으면 OOM, 너무 높으면 다른 process 여유 감소 | 학습 중·resume baseline에서 변경 금지 |

### `jax_memory_fraction: 0.80`의 의미

JAX가 GPU 메모리 pool로 예약할 수 있는 비율을 process 시작 전에 `XLA_PYTHON_CLIENT_MEM_FRACTION`으로 전달한다. 0.80은 항상 즉시 VRAM 80%를 실제 tensor로 사용한다는 뜻이 아니라 allocator가 사용할 pool 상한 설정이다.

- Loader의 **하드 범위**는 `0 < value <= 1`이다.
- 이 workspace의 **운영 guardrail**은 `0.50~0.95`로 둔다. 범위 밖이 parser 오류라는 뜻은 아니며, 현재 장비에서 안전값으로 간주하지 않는다는 뜻이다.
- 0.50에 가까우면 model·optimizer·activation이 pool에 들어가지 못해 OOM이 날 수 있다.
- 0.95에 가까우면 display, 다른 CUDA process, driver 여유가 작아질 수 있다.
- 이 값은 JAX import 전에만 적용된다. 실행 중 YAML을 고쳐도 현재 allocator는 바뀌지 않으며, 학습 중에는 절대로 수정하지 않는다.

현재 RTX 6000 Ada의 검증값은 0.80이다. 다른 값은 학습 process를 완전히 종료한 뒤 새 run에서 step 10→100→1000으로 다시 측정한다.

### 현재 PC에서 확인된 기준 `[측정]`

- batch 1, step 1000까지 loss·gradient가 finite였다.
- JAX used는 약 10.487GiB, peak는 약 15.846GiB로 안정됐다.
- 안정 구간 step은 약 0.225초, data decode는 약 0.048초였다.
- data 비중이 약 21.5%라 worker 증가가 첫 최적화 대상은 아니다.
- checkpoint 저장은 약 6.3~9.8초였고 host RSS가 저장 주기에 맞춰 상승했다. 장기 학습은 현재 `save_interval=1000`을 유지한다.

`num_train_steps`는 추가 횟수가 아니라 **최종 absolute step**이다. step 1000 checkpoint에서 `--target-step 30000`으로 resume하면 29,000 step을 더 실행한다. batch 1이면 30,000 step은 30,000 frame sample이며 201,999-frame dataset의 명목상 약 0.1485 epoch다.


## 4. Optimizer와 learning-rate는 현재 잠긴 mirror

아래 값은 YAML에 보이지만 현재 `training`처럼 조절하는 입력이 아니다. 사람이 공식 baseline을 확인할 수 있도록 적어둔 **locked reference mirror**이며 한 값이라도 바꾸면 `load_pi0_settings()`가 학습 전에 중단한다.

| 파라미터 | 현재값 `[코드]` | 쉬운 뜻 | 예상 영향 | 현재 변경 방법 |
|---|---:|---|---|---|
| warmup | 1,000 step | 작은 LR에서 peak까지 올리는 기간 | 너무 짧으면 초반 불안정 가능 `[추정]` | YAML 수정 불가 |
| peak LR | `2.5e-5` | 가장 큰 update 속도 | 크면 발산, 작으면 학습 부족 가능 `[추정]` | YAML 수정 불가 |
| decay steps | 30,000 | cosine 감소가 끝나는 위치 | 총 step과 학습 후반 동작 변경 | YAML 수정 불가 |
| decay LR | `2.5e-6` | 마지막 update 속도 | 후반 parameter 변화량 변경 | YAML 수정 불가 |
| AdamW `b1/b2` | `0.9 / 0.95` | gradient 이동평균 기억 정도 | update dynamics 변경 | YAML 수정 불가 |
| `eps` | `1e-8` | 수치 안정화 값 | 수치 동작 변경 | YAML 수정 불가 |
| weight decay | `1e-10` | 공식 baseline의 아주 작은 decay | OpenPI 주석상 0에서 OOM 사례 | YAML 수정 불가 |
| gradient clip | 1.0 | 큰 gradient update 제한 | 너무 작으면 둔화, 크면 발산 위험 `[추정]` | YAML 수정 불가 |

로그의 `grad_norm`은 clipping 전 값이므로 1.0보다 커도 바로 오류가 아니다. NaN/Inf 또는 여러 구간 동안 계속 커지는지를 본다. Step 1000은 warmup 종료점일 뿐 수렴 완료점이 아니다.

이 값을 실험하려면 새 run 이름만 바꾸면 안 된다. 별도 production profile과 config schema/validator를 구현해 실제 `TrainConfig.lr_schedule` 또는 `optimizer`를 명시적으로 생성하고, 기존 checkpoint를 resume하지 않은 fresh run에서 검증해야 한다. 현재 YAML 파일을 복사해 숫자만 바꾸는 방식은 의도적으로 실패한다.


## 5. 구조 실험 — 별도 production profile 구현이 먼저

Vision freeze, LoRA rank, horizon은 현재 YAML에서 선택 가능한 option이 아니라 `model_contract`의 locked mirror다. **새 run 이름만 만드는 것으로 부족하며**, production 코드가 새 구조를 실제로 만들고 검증하도록 별도 profile을 먼저 구현해야 한다.

| 실험 | 현재 상태 | 별도 구현해야 하는 부분 | 예상 영향·증상 | VRAM·속도 |
|---|---|---|---|---|
| Vision encoder 동결 | 414,803,696개 vision parameter 학습 | 새 freeze profile, trainable audit, optimizer state 검증 | 과적합 감소 또는 시각 적응력 부족 가능 `[추정]` | parameter+Adam logical 약 3.86GiB 절약 가능 |
| LoRA rank | PaliGemma rank 16, action expert rank 32 | 새 model variant/profile, weight merge·shape validator | capacity·checkpoint shape 변경 | LoRA 부분은 rank에 대략 선형, 실제 VRAM 재측정 |
| Base LLM full tune | 2,819,996,672개 base LLM 동결 | full-tune profile, precision·optimizer·memory 설계 | 첫 초기화/step OOM 가능 | persistent logical 약 26.3GiB 추가 추정 |
| Horizon·action dim | 50 / 32 | model config, v3 adapter, transform, norm stats, inference contract | loader/JIT/checkpoint shape 변경 | 늘리면 action compute·activation 증가 |
| Image·prompt | 224 / 48 | model transform, tokenizer/vision 검증, inference input | JIT·attention·vision 비용 변경 | 크기를 늘리면 compute 증가 |
| Action·카메라 의미 | 6 delta+gripper absolute, 2 active cameras | 새 data profile, asset ID, norm stats, inference 역변환 | loss 정상이어도 실제 로봇 실패 가능 | action 의미 자체는 VRAM 영향 작음 |
| EMA·precision·FSDP | None / bf16+fp32 / 1 | trainer·checkpoint·device topology profile | restore·수치·multi-device 오류 | EMA 약 +7GiB logical, FSDP 통신 비용 |

### 구조 실험의 통과 순서

1. 새 production profile 이름과 실제 model/data/optimizer 생성 코드를 구현한다.
2. 새 profile의 strict schema와 fail-fast validator를 구현한다.
3. 새 config 파일과 새 run 이름을 만든다.
4. action·dataset 의미가 바뀌면 새 asset ID와 norm stats를 만든다.
5. Abstract parameter count와 freeze filter를 다시 확인한다.
6. Step 1 forward/backward, checkpoint 저장, fresh restore를 확인한다.
7. 10→100→1000 안정성 뒤 긴 학습과 동일 조건 rollout을 비교한다.

현재 `pi0_piper_lora.yaml`의 optimizer, LR, vision/LoRA/horizon mirror를 직접 고치는 것은 실험 시작 방법이 아니다. Loader가 거부하는 것이 정상이다.


## 6. `config/training/pi0_piper_lora.yaml` 사용법

`scripts/training/train_from_config.py`를 사용할 때 `training`과 `runtime` 값의 source는 YAML이다. CLI에서 수치로 덮어쓸 수 있는 값은 이번 실행이 멈출 absolute step인 `--target-step`뿐이다. `--run-name`, `--resume`, `--check-only`, `--print-config`는 값 튜닝이 아니라 실행 모드다.

`scripts/training/train.py`는 wrapper가 최종적으로 위임하는 검증된 내부 trainer이며 기존 명령 호환을 위해 자체 CLI를 유지한다. 이 파일을 직접 실행하면 YAML이 source가 아니므로, 사람이 수행하는 정상 학습은 config wrapper를 사용한다.

### Config wrapper의 우선순위

```text
고정 production 계약                     변경 불가
        ↓
config/training/pi0_piper_lora.yaml               training/runtime source
        ↓
--target-step                            YAML final 이하의 임시 absolute stop
        ↓
--run-name / --resume / --check-only      실행 모드
```

### GPU 없이 resolved config만 출력

```bash
./scripts/training/train_from_config.py \
  --config config/training/pi0_piper_lora.yaml \
  --print-config
```

`--print-config`는 `--run-name` 없이도 실행할 수 있다. Strict YAML 검증과 resolved 값 출력만 수행하고 JAX/OpenPI, GPU, model weight, checkpoint manager를 사용하지 않는다.

출력의 `Locked contracts: PASS`는 YAML locked mirror가 `pi0_settings.py`의 baseline과 같다는 뜻이다. 실제 OpenPI model 검증은 학습/check-only로 내부 trainer에 위임될 때 별도로 실행된다.

### 현재 `r001`을 step 1000에서 30000까지 이어가기

```bash
cd /home/pc/vla_ws
conda activate /home/pc/vla_ws/.conda/env

./scripts/training/train_from_config.py \
  --config config/training/pi0_piper_lora.yaml \
  --run-name background_400_0818_v3_r001 \
  --resume \
  --target-step 30000
```

현재 `r001`에서는 YAML을 그대로 유지하고 `--target-step`만 사용한다. 기존 step 1000 checkpoint·optimizer state·normalization asset을 복원한다.

### 모델 weight와 run을 만들지 않는 data-contract 검사

```bash
./scripts/training/train_from_config.py \
  --config config/training/pi0_piper_lora.yaml \
  --run-name config_contract_check \
  --check-only
```

Check-only는 CPU를 강제하고 model weight와 run directory를 만들지 않아야 한다. 다만 OpenPI data transform까지 검증하므로 첫 노트북 code처럼 JAX/OpenPI 완전 미사용인 `--print-config`와 역할이 다르다.

### 새 실험 시작

```bash
./scripts/training/train_from_config.py \
  --config config/training/pi0_piper_lora.yaml \
  --run-name background_400_0818_v3_r002 \
  --target-step 1000
```

새 run에서는 `--resume`을 넣지 않는다. Locked optimizer/model 값을 바꾸는 구조 실험이라면 이 명령 전에 별도 production profile 구현이 필요하다.

다음 코드 셀은 명령을 실행하지 않고 문자열로만 만든다. 복사 전에 run 이름과 목표 step을 다시 읽는다.


In [ ]:
# production YAML의 workspace 상대 경로다.
CONFIG_ARGUMENT = "config/training/pi0_piper_lora.yaml"

# 현재 baseline run을 식별하는 이름이다.
BASELINE_RUN_NAME = "background_400_0818_v3_r001"

# baseline 학습이 도달할 최종 absolute step이다.
BASELINE_TARGET_STEP = 30_000

# shell에서 복사할 resume 명령 문자열이다. 이 셀은 실행하지 않는다.
RESUME_COMMAND = (
    "./scripts/training/train_from_config.py "
    f"--config {CONFIG_ARGUMENT} "
    f"--run-name {BASELINE_RUN_NAME} "
    "--resume "
    f"--target-step {BASELINE_TARGET_STEP}"
)

# shell에서 복사할 check-only 명령 문자열이다. 이 셀은 실행하지 않는다.
CHECK_ONLY_COMMAND = (
    "./scripts/training/train_from_config.py "
    f"--config {CONFIG_ARGUMENT} "
    "--run-name config_contract_check "
    "--check-only"
)

# GPU 없이 resolved YAML 값만 표시할 명령 문자열이다. 이 셀은 실행하지 않는다.
PRINT_CONFIG_COMMAND = (
    "./scripts/training/train_from_config.py "
    f"--config {CONFIG_ARGUMENT} "
    "--print-config"
)

print("Resume preview    :", RESUME_COMMAND)
print("Print-config      :", PRINT_CONFIG_COMMAND)
print("Check-only preview:", CHECK_ONLY_COMMAND)
print("Subprocess executed: NO")


## 7. Resume와 새 run 결정표

| 하려는 변경 | 현재 `r001` resume | 새 run | 추가 구현 | 이유 |
|---|---:|---:|---:|---|
| 목표 step만 1000→30000 | 가능 | 선택 | 불필요 | model·optimizer·data 계약 동일 |
| log/save/keep/worker | 현재는 변경하지 않음 | 운영 비교면 권장 | 불필요 | 구조는 같지만 baseline 조건·성능이 달라짐 |
| batch size·seed | 금지 | 권장 | 불필요 | sample 수, JIT shape, shuffle/noise 변경 |
| dataset path·asset ID | 금지 | 필수 | 새 stats가 필요할 수 있음 | 데이터 내용·normalization 계약 변경 |
| learning rate·optimizer | 금지 | 필수 | 별도 optimizer profile 필수 | YAML locked mirror, update 규칙 변경 |
| vision freeze·LoRA rank | 금지 | 필수 | 별도 model/freeze profile 필수 | trainable parameter·Adam state 변경 |
| horizon·image·token·action 의미 | 금지 | 필수 | model+data+inference profile 필수 | shape 또는 물리적 의미 변경 |

### 현재 resume 검증의 한계

Production은 checkpoint normalization asset과 현재 norm stats가 같은지 검사하고 pinned model contract도 검증한다. 하지만 아직 다음 정보를 checkpoint metadata와 자동 비교하지 않는다.

- 사용했던 YAML 파일의 SHA-256
- dataset directory 내용 또는 manifest hash
- 이전 batch size와 현재 batch size
- 이전 seed와 현재 seed
- 이전 worker/log/save 운영값

따라서 loader가 새 YAML 자체를 통과시킨다고 해서 기존 checkpoint와 같은 실험이라는 뜻은 아니다. 현재 `r001`은 검증 당시의 YAML을 그대로 유지하고 `--target-step`만 바꾼다. **Dataset, asset ID, batch size, seed를 바꾼 뒤 `r001 --resume`을 실행하지 않는다.**

Resume 전에는 `--print-config`로 resolved 값을 확인하고 첫 셀의 config SHA-256을 실험 기록에 남긴다. 완전 자동 보호가 필요하면 향후 checkpoint metadata에 config hash, dataset fingerprint, batch, seed를 저장하고 resume 때 비교하는 기능을 별도로 구현해야 한다.

Loader iterator 위치도 checkpoint되지 않는다. Resume하면 `seed + latest_step`으로 새 shuffle stream을 시작하므로 model·optimizer는 이어져도 sample 순서가 byte 단위로 완전히 이어지는 것은 아니다.


## 8. 이상 증상에서 원인 찾기

| 증상 | 먼저 확인할 값 | 바로 하지 말아야 할 일 |
|---|---|---|
| 첫 step OOM | batch, vision freeze, EMA, full fine-tune 여부 | 여러 구조 값을 동시에 바꾸기 |
| loss NaN/Inf, grad 지속 급증 | LR, normalization, 데이터 유한값 | checkpoint를 계속 덮어쓰기 |
| `avg_data_time_s`가 compute보다 큼 | worker 수, MP4 decode, CPU/RAM | model 구조부터 바꾸기 |
| checkpoint 때 긴 정지·RAM spike | save interval, 디스크 속도·여유 | save를 지나치게 자주 하기 |
| 디스크가 계속 증가 | keep period와 보존 step | run directory 수동 일부 삭제 |
| loss는 내려가지만 로봇이 실패 | held-out episode, 카메라/action/normalization 계약 | 무조건 step만 늘리기 |
| restore 실패 | run 이름, latest step, model/freeze/optimizer 계약 | base weight와 checkpoint를 임의 혼합 |

현재 production metric은 training loss다. Validation split이나 실제 로봇 성공률을 대신하지 않는다. 좋은 파라미터는 training graph만 보고 고르는 것이 아니라 held-out episode와 저속 실제 rollout으로 확인해야 한다.


## 9. 실험 기록 템플릿

설정을 바꾸기 전에 아래 항목을 먼저 채운다. 결과를 본 뒤 가설을 쓰면 원래 예상과 실제 결과를 구분할 수 없다.

필수 기록:

1. 왜 이 값 하나를 바꾸는지
2. baseline과 실험 run 이름
3. 고정한 dataset·asset·seed·총 sample 수
4. 예상하는 loss·속도·메모리·성공률 변화
5. 실제 step 10/100/1000 및 최종 결과

다음 셀은 기록 템플릿을 메모리에만 만들고 YAML 문자열로 보여준다. 파일을 생성하거나 수정하지 않는다. 출력 내용을 복사해 별도 실험 기록 파일에 저장할 수 있다.


In [ ]:
# 한 번에 하나의 변경만 기록하는 실험 템플릿이다.
EXPERIMENT_RECORD = {
    "identity": {
        "date": "YYYY-MM-DD",
        "owner": "",
        "baseline_run": "background_400_0818_v3_r001",
        "experiment_run": "background_400_0818_v3_rXXX",
        "config_file": "config/training/pi0_piper_lora.yaml",
        "config_sha256": CONFIG_SHA256,
    },
    "hypothesis": {
        "one_changed_parameter": "",
        "baseline_value": "",
        "experiment_value": "",
        "reason_before_training": "",
        "expected_loss_change": "",
        "expected_vram_change": "",
        "expected_robot_success_change": "",
    },
    "fixed_conditions": {
        "dataset_root": "data/datasets/pick_green_to_orange/background_400_0818_v3",
        "dataset_manifest_sha256": "",
        "asset_id": "background_400_0818_v3",
        "norm_stats_sha256": "",
        "seed": 42,
        "batch_size": 1,
        "target_step": 30000,
        "evaluation_episode_ids": [],
        "robot_initial_condition": "",
    },
    "training_results": {
        "step_10": {"status": "", "loss": None, "grad_norm": None},
        "step_100": {"status": "", "loss": None, "grad_norm": None},
        "step_1000": {"status": "", "loss": None, "grad_norm": None},
        "final": {"step": None, "loss": None, "grad_norm": None},
        "gpu_peak_gib": None,
        "host_peak_gib": None,
        "median_step_seconds": None,
        "checkpoint_status": "",
    },
    "evaluation_results": {
        "held_out_episodes": "",
        "robot_successes": None,
        "robot_trials": None,
        "safety_events": [],
        "observations": "",
    },
    "decision": {
        "keep_or_reject": "",
        "evidence": "",
        "next_single_change": "",
    },
}

# 복사 가능한 YAML 형태의 실험 기록 문자열이다.
EXPERIMENT_RECORD_PREVIEW = yaml.safe_dump(
    EXPERIMENT_RECORD,
    allow_unicode=True,
    sort_keys=False,
)

print(EXPERIMENT_RECORD_PREVIEW)
print("File write: NOT USED")


## 10. 변경 전 최종 체크리스트

파라미터 하나를 바꾸기 전에 아래 질문에 모두 답한다.

- [ ] `train_pi0_from_config.py`와 `train_pi0.py` 중 어느 진입점을 쓰는지 구분했는가?
- [ ] 이 값은 training/runtime, locked mirror, 별도 production profile 중 어디에 속하는가?
- [ ] Loader의 hard guardrail 통과와 실제 GPU 안전성을 구분했는가?
- [ ] Optimizer/LR/vision/LoRA/horizon 변경에 별도 profile 구현 계획이 있는가?
- [ ] 새 run 이름만으로 구조 실험이 가능하다고 오해하지 않았는가?
- [ ] Resume 전에 resolved config와 config SHA-256을 기록했는가?
- [ ] 현재 `r001`에서는 YAML을 그대로 두고 `--target-step`만 바꾸는가?
- [ ] Dataset, asset ID, batch, seed 변경 뒤 기존 checkpoint를 resume하지 않는가?
- [ ] `jax_memory_fraction=0.80`을 학습 중 수정하지 않는가?
- [ ] 한 번에 하나만 바꾸고 동일한 평가 episode를 사용하는가?
- [ ] Step 10→100→1000, checkpoint restore, 실제 rollout 계획이 있는가?

현재 기준의 첫 선택은 단순하다. `r001`은 YAML과 고정 계약을 그대로 유지한 채 `--target-step 30000`으로 baseline을 이어간다. 실제 rollout까지 기록한 뒤, 별도 production profile을 구현한 새 run에서 vision encoder 동결처럼 한 가지 구조 변경만 비교한다.
